# Experiment 5.3.2 — Supervised causal WHEN representation benchmark

Analysis-only notebook. The experiment first asks whether the temporal branch contains usable WHEN before any WHAT × WHEN fusion is attempted.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')

ROOT = find_repo_root()
OUT = ROOT / 'notebooks/artifacts/experiment_5_3_2_when_representation/supervised_causal_when_v1'
manifest = json.loads((OUT / 'manifest.json').read_text(encoding='utf-8'))
runs = pd.read_csv(OUT / 'runs.csv')
histories = pd.read_csv(OUT / 'histories.csv')
probe_runs = pd.read_csv(OUT / 'probe_runs.csv')
group_probe_runs = pd.read_csv(OUT / 'group_probe_runs.csv')
ablation_runs = pd.read_csv(OUT / 'ablation_runs.csv')
baseline_runs = pd.read_csv(OUT / 'baseline_runs.csv')
local_reference = pd.read_csv(OUT / 'local_reference.csv')
manifest


## 1. Primary WHEN quality
Primary interface: post-reset membrane $U_t$. Rank models by ordered phase BA, then causal history attribution and progress MAE.


In [ ]:
condition_order = [
    'mem_single_s4', 'mem_single_s5', 'mem_single_s6', 'mem_multi_s456',
    'syn_single_s4', 'syn_single_s5', 'syn_single_s6', 'syn_multi_s456',
    'rsnn_shortmem',
]
u_probe = probe_runs[probe_runs['feature_type'].eq('membrane')].copy()
native_summary = runs.groupby(['condition', 'family'], as_index=False).agg(
    native_phase_ba_mean=('native_test_phase_balanced_accuracy', 'mean'),
    native_phase_ba_sd=('native_test_phase_balanced_accuracy', 'std'),
    native_progress_mae_mean=('native_test_progress_sample_balanced_mae', 'mean'),
    native_progress_mae_sd=('native_test_progress_sample_balanced_mae', 'std'),
)
probe_summary = u_probe.groupby(['condition', 'family'], as_index=False).agg(
    probe_u_phase_ba_mean=('phase_probe_test_ba', 'mean'),
    probe_u_phase_ba_sd=('phase_probe_test_ba', 'std'),
    probe_u_progress_mae_mean=('progress_probe_test_mae', 'mean'),
    probe_u_progress_mae_sd=('progress_probe_test_mae', 'std'),
)
primary = native_summary.merge(probe_summary, on=['condition', 'family'])
primary['condition'] = pd.Categorical(primary['condition'], condition_order, ordered=True)
primary = primary.sort_values('condition')
primary


In [ ]:
baseline_summary = baseline_runs.groupby('baseline', as_index=False).agg(
    phase_ba_mean=('phase_probe_test_ba', 'mean'),
    phase_ba_sd=('phase_probe_test_ba', 'std'),
    progress_mae_mean=('progress_probe_test_mae', 'mean'),
    progress_mae_sd=('progress_probe_test_mae', 'std'),
)
baseline_summary


In [ ]:
plot_df = primary.set_index('condition').reindex(condition_order).reset_index()
fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(plot_df))
ax.errorbar(x, plot_df['probe_u_phase_ba_mean'], yerr=plot_df['probe_u_phase_ba_sd'], marker='o', capsize=3)
for _, row in baseline_summary.iterrows():
    ax.axhline(row['phase_ba_mean'], linestyle='--', label=row['baseline'])
ax.axhline(0.10, linestyle=':', label='10% chance')
ax.set_xticks(x, plot_df['condition'], rotation=30, ha='right')
ax.set_ylabel('Test phase balanced accuracy')
ax.set_title('Exp5.3.2: causal WHEN accessibility from membrane U_t')
ax.legend()
fig.tight_layout()
plt.show()


## 2. Membrane-vs-synaptic timescale sweeps
Compare 242/492/992 ms while changing only one memory location at a time.


In [ ]:
single_map = {
    'mem_single_s4': ('mem', 242), 'mem_single_s5': ('mem', 492), 'mem_single_s6': ('mem', 992),
    'syn_single_s4': ('syn', 242), 'syn_single_s5': ('syn', 492), 'syn_single_s6': ('syn', 992),
}
sweep = u_probe[u_probe['condition'].isin(single_map)].copy()
sweep[['memory_location', 'tau_ms']] = sweep['condition'].apply(lambda name: pd.Series(single_map[name]))
sweep_summary = sweep.groupby(['memory_location', 'tau_ms'], as_index=False).agg(
    mean_ba=('phase_probe_test_ba', 'mean'), sd_ba=('phase_probe_test_ba', 'std'))
fig, ax = plt.subplots(figsize=(7.5, 5.0))
for memory_location, frame in sweep_summary.groupby('memory_location'):
    frame = frame.sort_values('tau_ms')
    ax.errorbar(frame['tau_ms'], frame['mean_ba'], yerr=frame['sd_ba'], marker='o', capsize=3, label=memory_location)
ax.set_xlabel('Long memory time constant (ms)')
ax.set_ylabel('Membrane-probe test phase BA')
ax.set_title('Where should long temporal memory live?')
ax.legend()
fig.tight_layout()
plt.show()


## 3. Internal state to spike-code accessibility
A useful hardware-oriented WHEN branch should ideally retain phase information in instantaneous or short trailing spike windows.


In [ ]:
feature_order = ['membrane', 'synaptic', 'spike', 'spike250', 'spike500']
access = probe_runs.groupby(['condition', 'feature_type'], as_index=False).agg(
    mean_ba=('phase_probe_test_ba', 'mean'), sd_ba=('phase_probe_test_ba', 'std'))
pivot = access.pivot(index='condition', columns='feature_type', values='mean_ba').reindex(condition_order)
pivot[feature_order]


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(condition_order))
width = 0.16
for offset, feature in enumerate(['membrane', 'spike', 'spike250', 'spike500']):
    values = pivot[feature].to_numpy()
    ax.bar(x + (offset - 1.5) * width, values, width=width, label=feature)
ax.set_xticks(x, condition_order, rotation=30, ha='right')
ax.set_ylabel('Test phase BA')
ax.set_title('WHEN accessibility: internal membrane vs spike code')
ax.legend()
fig.tight_layout()
plt.show()


## 4. Causal history attribution
Ordered performance alone is insufficient. Compare ordered, state_reset, and temporal_shuffle to determine whether WHEN truly depends on history.


In [ ]:
abl = ablation_runs.copy()
abl_summary = abl.groupby(['condition', 'ablation'], as_index=False).agg(
    phase_ba=('probe_test_phase_ba', 'mean'),
    phase_ba_sd=('probe_test_phase_ba', 'std'),
    progress_mae=('probe_test_progress_mae', 'mean'),
)
ordered = abl[abl['ablation'].eq('ordered')][['condition', 'seed', 'probe_test_phase_ba']].rename(columns={'probe_test_phase_ba': 'ordered_ba'})
reset = abl[abl['ablation'].eq('state_reset')][['condition', 'seed', 'probe_test_phase_ba']].rename(columns={'probe_test_phase_ba': 'reset_ba'})
shuffle = abl[abl['ablation'].eq('temporal_shuffle')].groupby(['condition', 'seed'], as_index=False)['probe_test_phase_ba'].mean().rename(columns={'probe_test_phase_ba': 'shuffle_ba'})
history_gain = ordered.merge(reset, on=['condition', 'seed']).merge(shuffle, on=['condition', 'seed'])
history_gain['H_reset'] = history_gain['ordered_ba'] - history_gain['reset_ba']
history_gain['H_shuffle'] = history_gain['ordered_ba'] - history_gain['shuffle_ba']
history_gain.groupby('condition', as_index=False).agg(
    H_reset_mean=('H_reset', 'mean'), H_reset_sd=('H_reset', 'std'),
    H_shuffle_mean=('H_shuffle', 'mean'), H_shuffle_sd=('H_shuffle', 'std'),
).set_index('condition').reindex(condition_order)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
for ablation in ['ordered', 'state_reset', 'temporal_shuffle']:
    frame = abl_summary[abl_summary['ablation'].eq(ablation)].set_index('condition').reindex(condition_order)
    ax.plot(np.arange(len(condition_order)), frame['phase_ba'], marker='o', label=ablation)
ax.set_xticks(np.arange(len(condition_order)), condition_order, rotation=30, ha='right')
ax.set_ylabel('Membrane-probe test phase BA')
ax.set_title('Causal attribution of WHEN')
ax.legend()
fig.tight_layout()
plt.show()


## 5. Multi-tau subgroup diagnostics
Inspect which 242/492/992 ms subgroup carries phase/progress information inside the multi-memory conditions.


In [ ]:
group_summary = group_probe_runs.groupby(['condition', 'group'], as_index=False).agg(
    phase_ba_mean=('phase_probe_test_ba', 'mean'),
    phase_ba_sd=('phase_probe_test_ba', 'std'),
    progress_mae_mean=('progress_probe_test_mae', 'mean'),
)
group_summary


## 6. Validation learning curves
Checkpoint selection is validation-only: maximize native U phase BA, then lower progress MAE, then lower total loss.


In [ ]:
curve = histories.groupby(['condition', 'epoch'], as_index=False).agg(
    mean_val_phase_ba=('val_phase_balanced_accuracy', 'mean'),
    sd_val_phase_ba=('val_phase_balanced_accuracy', 'std'),
)
fig, ax = plt.subplots(figsize=(9.5, 5.5))
for condition in condition_order:
    frame = curve[curve['condition'].eq(condition)]
    ax.plot(frame['epoch'], frame['mean_val_phase_ba'], label=condition)
    ax.fill_between(frame['epoch'], frame['mean_val_phase_ba'] - frame['sd_val_phase_ba'], frame['mean_val_phase_ba'] + frame['sd_val_phase_ba'], alpha=0.12)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation phase BA')
ax.set_title('Exp5.3.2 validation WHEN learning')
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
plt.show()
